# SENTINEL QLoRA fine-tune on Kaggle (free T4)

Same `train_qlora.py` as on the RTX 4060, run on a Kaggle T4 16 GB so the local PC does no training.

Before **Save & Run All**:
1. Settings → Accelerator: **GPU T4 x2** (one GPU is used) · Internet: **On** (needs a phone-verified account).
2. Add Input → your private dataset `sentinel-sft` with `train_qlora.py`, `sft_train.jsonl`, `sft_val.jsonl`.
3. To resume a run that hit the time budget: also Add Input → the output of the previous version of this notebook.

The base model is downloaded from Hugging Face at the same revision as `D:\ai-models\Qwen3.5-4B`, so nothing large is uploaded.
Output: `/kaggle/working/sentinel-lora-r2-adapter.zip` (adapter only, ~120 MB) → bench on the M5 against `qwen3.5:4b`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # GPU T4 x2: the script trains on a single GPU

BASE_REPO = "Qwen/Qwen3.5-4B"
BASE_REV = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"  # snapshot of D:\ai-models\Qwen3.5-4B
RUN = "sentinel-lora-r2"
EPOCHS = 2
MAX_LEN = 5120  # longest train sample is 4878 tokens, so nothing is dropped
TIME_BUDGET_H = 11.0  # Kaggle kills a session at 12 h; stop before that so checkpoints are saved

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q transformers==5.20.2 peft==0.21.2 bitsandbytes==0.50.2 accelerate==1.15.0

In [ ]:
import importlib.metadata as md
import torch
for p in ("torch", "transformers", "peft", "bitsandbytes", "accelerate"):
    print(p, md.version(p))
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2"
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
import shutil
from pathlib import Path

inp, working = Path("/kaggle/input"), Path("/kaggle/working")
work, out = working / "finetune", working / RUN
(work / "data").mkdir(parents=True, exist_ok=True)
shutil.copy(next(inp.rglob("train_qlora.py")), work)
for name in ("sft_train.jsonl", "sft_val.jsonl"):
    shutil.copy(next(inp.rglob(name)), work / "data" / name)

# Resume: take the latest checkpoint from a previous version's output attached as input.
prev = sorted(inp.rglob(f"{RUN}/checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
if prev and not any(out.glob("checkpoint-*")):
    shutil.copytree(prev[-1], out / prev[-1].name)
    print("resuming from", prev[-1])
print(sorted(p.relative_to(working).as_posix() for p in working.rglob("*") if p.is_file()))

In [ ]:
from huggingface_hub import snapshot_download

base = snapshot_download(BASE_REPO, revision=BASE_REV, local_dir="/tmp/Qwen3.5-4B")
print(base)

In [ ]:
import subprocess, sys, time

cmd = [sys.executable, "-u", str(work / "train_qlora.py"), "--base", base, "--out", str(out),
       "--epochs", str(EPOCHS), "--max-len", str(MAX_LEN), "--resume"]
log_path = working / f"{RUN}.log"
deadline = time.time() + TIME_BUDGET_H * 3600
with log_path.open("a", encoding="utf-8") as log:
    proc = subprocess.Popen(cmd, cwd=work, stdout=log, stderr=subprocess.STDOUT)
    seen = 0
    while proc.poll() is None:
        time.sleep(60)
        text = log_path.read_text(encoding="utf-8", errors="replace")
        new, seen = text[seen:], len(text)
        lines = [l for l in new.replace("\r", "\n").split("\n") if l.strip()]
        if lines:
            print(*lines[-3:], sep="\n", flush=True)
        if time.time() > deadline:
            proc.terminate()
            proc.wait()
            print(f"Time budget hit: re-run with this version's output as input to resume.")
            break
print("exit code", proc.returncode)

In [ ]:
import json, zipfile

state = sorted(out.glob("checkpoint-*/trainer_state.json"), key=lambda p: int(p.parent.name.split("-")[1]))
if state:
    for row in json.loads(state[-1].read_text())["log_history"]:
        print(row)

if (out / "adapter_model.safetensors").exists():
    zip_path = working / f"{RUN}-adapter.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for f in out.iterdir():
            if f.is_file():
                z.write(f, f"{RUN}/{f.name}")
    print("download:", zip_path, f"{zip_path.stat().st_size / 2**20:.0f} MB")
else:
    print("Not finished: checkpoints are in the output; attach it as input and run again to resume.")